In [ ]:
import os
import time
import urllib.parse
from datetime import datetime

import pandas as pd
from bs4 import BeautifulSoup as bs
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options


# =========================================================
# 0. 사용자 설정 영역
# =========================================================

# ---------------------------------------------------------
# 수집 담당자
# ---------------------------------------------------------
COLLECTOR_NAME = "JKS"


# ---------------------------------------------------------
# 카페 정보
# ---------------------------------------------------------
CAFE_NAME = "대구맘365"
CAFE_ID = 24000254

# 수집할 게시판 ID
MENU_IDS = [
    13,
]


# ---------------------------------------------------------
# 검색 키워드
# ---------------------------------------------------------
KEYWORDS = [
    "위생",
    "청결",
    "청소",
    "세탁",
    "빨래",
    "세척",
    "소독",
    "살균",
    "환기",
    "공기",
    "건조",
    "냄새",
]


# ---------------------------------------------------------
# 검색 기간
# YYYYMMDD
# ---------------------------------------------------------
START_DATE = "20250101"
END_DATE = "20260901"


# ---------------------------------------------------------
# 페이지 설정
# ---------------------------------------------------------
MAX_PAGE = 20
PAGE_SIZE = 50


# ---------------------------------------------------------
# 페이지 로딩 대기 시간
# 너무 짧으면 게시물이 제대로 로딩되지 않을 수 있음
# ---------------------------------------------------------
LIST_WAIT = 1.5
ARTICLE_WAIT = 1.0


# ---------------------------------------------------------
# Selenium 전용 Chrome 프로필
#
# 중요:
# 평소 사용하는 Chrome 프로필이 아니라
# Selenium 전용 프로필을 따로 사용한다.
#
# 최초 1회 Selenium 창에서 직접 네이버 로그인 필요
# 이후 로그인 세션이 해당 폴더에 저장됨
# ---------------------------------------------------------
CHROME_PROFILE_PATH = os.path.abspath(
    r"C:\selenium_profile\naver_cafe"
)


# ---------------------------------------------------------
# 결과 저장 폴더
# ---------------------------------------------------------
OUTPUT_DIR = "./crawling_results"


# =========================================================
# 1. Chrome Driver 생성
# =========================================================

def create_driver():

    os.makedirs(
        CHROME_PROFILE_PATH,
        exist_ok=True
    )

    options = Options()

    # -----------------------------------------------------
    # 로그인 유지 핵심
    # -----------------------------------------------------
    options.add_argument(
        f"--user-data-dir={CHROME_PROFILE_PATH}"
    )

    options.add_argument(
        "--profile-directory=Default"
    )

    # Chrome 최대화
    options.add_argument(
        "--start-maximized"
    )

    # Selenium 자동화 안내 일부 제거
    options.add_experimental_option(
        "excludeSwitches",
        ["enable-automation"]
    )

    options.add_experimental_option(
        "useAutomationExtension",
        False
    )

    driver = wb.Chrome(
        options=options
    )

    driver.implicitly_wait(5)

    return driver


# =========================================================
# 2. 네이버 로그인 확인
# =========================================================

def check_naver_login(driver):

    print()
    print("=" * 60)
    print("네이버 로그인 상태 확인")
    print("=" * 60)

    driver.get(
        "https://www.naver.com/"
    )

    time.sleep(2)

    # -----------------------------------------------------
    # 네이버 메인에서 로그인 버튼 탐색
    #
    # 네이버 DOM 변경 가능성이 있으므로
    # 여러 selector를 확인
    # -----------------------------------------------------

    login_selectors = [
        'a[href*="nidlogin.login"]',
        'a.MyView-module__link_login___HpHMW',
        '.link_login',
    ]

    login_required = False

    for selector in login_selectors:

        elements = driver.find_elements(
            By.CSS_SELECTOR,
            selector
        )

        if elements:
            login_required = True
            break

    # -----------------------------------------------------
    # 로그인이 안 되어 있는 경우
    # -----------------------------------------------------

    if login_required:

        print()
        print("네이버 로그인이 필요합니다.")
        print()
        print(
            "현재 열린 Selenium Chrome 창에서 "
            "네이버에 로그인해주세요."
        )
        print()
        print(
            "가능하면 '로그인 상태 유지'를 체크하세요."
        )
        print()
        print(
            "로그인 완료 후 이 콘솔로 돌아와 "
            "Enter를 눌러주세요."
        )
        print()

        input("로그인 완료 후 Enter > ")

        time.sleep(2)

        # 로그인 확인
        driver.get(
            "https://www.naver.com/"
        )

        time.sleep(2)

        print()
        print("로그인 완료 후 크롤링을 진행합니다.")

    else:

        print(
            "네이버 로그인 세션이 유지되고 있습니다."
        )


# =========================================================
# 3. 검색 URL 생성
# =========================================================

def make_search_url(
    cafe_id,
    menu_id,
    keyword,
    page,
    start_date,
    end_date,
):

    # 한글 검색어 URL Encoding
    q = urllib.parse.quote(
        keyword
    )

    url = (
        f"https://cafe.naver.com/f-e/cafes/"
        f"{cafe_id}/menus/{menu_id}"
        f"?viewType=L"
        f"&page={page}"
        f"&size={PAGE_SIZE}"
        f"&ta=ARTICLE_COMMENT"
        f"&from={start_date}"
        f"&to={end_date}"
        f"&q={q}"
    )

    return url


# =========================================================
# 4. 로그인/접근 상태 확인
# =========================================================

def check_page_status(driver):

    current_url = driver.current_url.lower()

    page_source = driver.page_source

    # -----------------------------------------------------
    # 로그인 페이지로 이동한 경우
    # -----------------------------------------------------

    if (
        "nidlogin.login" in current_url
        or "nid.naver.com" in current_url
    ):

        raise RuntimeError(
            "네이버 로그인 세션이 만료되었습니다."
        )

    # -----------------------------------------------------
    # 접근 제한 / 비정상 페이지 간단 체크
    # -----------------------------------------------------

    error_keywords = [
        "접근이 제한",
        "일시적으로 이용이 제한",
        "비정상적인 접근",
    ]

    for keyword in error_keywords:

        if keyword in page_source:

            raise RuntimeError(
                f"네이버 접근 제한 감지: {keyword}"
            )


# =========================================================
# 5. 게시글 URL 수집
# =========================================================

def collect_article_urls(driver):

    # -----------------------------------------------------
    # key = URL
    #
    # 같은 게시물이 여러 검색어에서 발견될 수 있으므로
    # URL 기준으로 중복 제거
    #
    # matched_keywords에는
    # 어떤 검색어에 걸렸는지 모두 저장
    # -----------------------------------------------------

    article_dict = {}

    for menu_id in MENU_IDS:

        for keyword in KEYWORDS:

            print()
            print("=" * 60)

            print(
                f"[검색 시작]"
                f" 카페={CAFE_NAME}"
                f" / 게시판={menu_id}"
                f" / 검색어={keyword}"
            )

            print(
                f"기간={START_DATE}"
                f" ~ {END_DATE}"
            )

            print("=" * 60)

            for page in range(
                1,
                MAX_PAGE + 1
            ):

                search_url = make_search_url(
                    cafe_id=CAFE_ID,
                    menu_id=menu_id,
                    keyword=keyword,
                    page=page,
                    start_date=START_DATE,
                    end_date=END_DATE,
                )

                driver.get(
                    search_url
                )

                time.sleep(
                    LIST_WAIT
                )

                check_page_status(
                    driver
                )

                # -------------------------------------------------
                # 게시글 링크
                # -------------------------------------------------

                a_tags = driver.find_elements(
                    By.CSS_SELECTOR,
                    ".article"
                )

                # -------------------------------------------------
                # 검색 결과가 더 이상 없는 경우
                # -------------------------------------------------

                if not a_tags:

                    print(
                        f"[종료]"
                        f" 게시판={menu_id}"
                        f" / 검색어={keyword}"
                        f" / {page}페이지 결과 없음"
                    )

                    break

                new_count = 0

                for tag in a_tags:

                    href = tag.get_attribute(
                        "href"
                    )

                    if not href:
                        continue

                    # -------------------------------------------------
                    # 새로운 게시물
                    # -------------------------------------------------

                    if href not in article_dict:

                        article_dict[href] = {
                            "url": href,
                            "menu_id": menu_id,
                            "matched_keywords": set(),
                        }

                        new_count += 1

                    # -------------------------------------------------
                    # 해당 게시물이 걸린 검색어 저장
                    # -------------------------------------------------

                    article_dict[href][
                        "matched_keywords"
                    ].add(keyword)

                print(
                    f"{page}페이지 완료"
                    f" / 신규={new_count}"
                    f" / 누적={len(article_dict)}"
                )

    # -----------------------------------------------------
    # set → 문자열 변환
    # -----------------------------------------------------

    article_data = []

    for data in article_dict.values():

        article_data.append({
            "url":
                data["url"],

            "menu_id":
                data["menu_id"],

            "matched_keywords":
                "|".join(
                    sorted(
                        data["matched_keywords"]
                    )
                ),
        })

    return article_data


# =========================================================
# 6. 게시글 상세 수집
# =========================================================

def collect_article_detail(
    driver,
    article_info,
):

    url = article_info[
        "url"
    ]

    driver.get(
        url
    )

    time.sleep(
        ARTICLE_WAIT
    )

    check_page_status(
        driver
    )

    # -----------------------------------------------------
    # 구버전 네이버 카페 대응
    # -----------------------------------------------------

    try:

        driver.switch_to.frame(
            "cafe_main"
        )

    except Exception:

        pass

    soup = bs(
        driver.page_source,
        "html.parser"
    )

    # -----------------------------------------------------
    # 제목
    # -----------------------------------------------------

    title_el = soup.select_one(
        ".title_text"
    )

    title = (
        title_el.get_text(
            strip=True
        )
        if title_el
        else ""
    )

    # -----------------------------------------------------
    # 날짜
    # -----------------------------------------------------

    date_el = soup.select_one(
        ".date"
    )

    date = (
        date_el.get_text(
            strip=True
        )
        if date_el
        else ""
    )

    # -----------------------------------------------------
    # 본문
    # -----------------------------------------------------

    content_el = (
        soup.select_one(
            ".se-main-container"
        )
        or soup.select_one(
            ".se-module"
        )
    )

    content = (
        content_el.get_text(
            "\n",
            strip=True
        )
        if content_el
        else ""
    )

    # -----------------------------------------------------
    # 댓글
    # -----------------------------------------------------

    comment_els = soup.select(
        "span.text_comment"
    )

    comments = " | ".join(
        c.get_text(
            strip=True
        )
        for c in comment_els
    )

    comment_count = len(
        comment_els
    )

    # -----------------------------------------------------
    # 결과
    # -----------------------------------------------------

    return {

        "cafe_name":
            CAFE_NAME,

        "cafe_id":
            CAFE_ID,

        "menu_id":
            article_info[
                "menu_id"
            ],

        "matched_keywords":
            article_info[
                "matched_keywords"
            ],

        "title":
            title,

        "date":
            date,

        "content":
            content,

        "comments":
            comments,

        "comment_count":
            comment_count,

        "url":
            url,

        "collector":
            COLLECTOR_NAME,
    }


# =========================================================
# 7. 전체 게시글 상세 수집
# =========================================================

def collect_all_articles(
    driver,
    article_data,
):

    rows = []

    total = len(
        article_data
    )

    for idx, article_info in enumerate(
        article_data,
        1
    ):

        try:

            row = collect_article_detail(
                driver,
                article_info
            )

            rows.append(
                row
            )

            print(
                f"[{idx}/{total}] "
                f"{row['title'][:40]}"
            )

        except RuntimeError as e:

            # -------------------------------------------------
            # 로그인 만료나 접근제한은
            # 계속 진행하지 않고 중단
            # -------------------------------------------------

            print()
            print("=" * 60)
            print("크롤링 중단")
            print(e)
            print("=" * 60)

            break

        except Exception as e:

            print(
                f"[오류]"
                f" {idx}/{total}"
                f" {article_info['url']}"
                f" → {e}"
            )

        finally:

            driver.switch_to.default_content()

    return rows


# =========================================================
# 8. 결과 저장
# =========================================================
def save_results(rows):

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True
    )

    # -----------------------------------------------------
    # 팀 공통 컬럼 순서
    # -----------------------------------------------------
    columns = [
        "cafe_name",
        "cafe_id",
        "menu_id",
        "matched_keywords",
        "title",
        "date",
        "content",
        "comments",
        "comment_count",
        "url",
        "collector",
    ]

    df = pd.DataFrame(
        rows,
        columns=columns
    )

    collected_at = datetime.now().strftime(
        "%Y%m%d_%H%M"
    )

    filename_base = (
        f"{CAFE_NAME}"
        f"_{START_DATE}"
        f"_{END_DATE}"
        f"_{COLLECTOR_NAME}"
        f"_{collected_at}"
    )

    # -----------------------------------------------------
    # 저장 경로
    # -----------------------------------------------------
    csv_path = os.path.join(
        OUTPUT_DIR,
        f"{filename_base}.csv"
    )

    pkl_path = os.path.join(
        OUTPUT_DIR,
        f"{filename_base}.pkl"
    )

    jsonl_path = os.path.join(
        OUTPUT_DIR,
        f"{filename_base}.jsonl"
    )

    # -----------------------------------------------------
    # CSV 저장
    # -----------------------------------------------------
    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    # -----------------------------------------------------
    # PKL 저장
    # -----------------------------------------------------
    df.to_pickle(
        pkl_path
    )

    # -----------------------------------------------------
    # JSONL 저장
    #
    # 게시글 1개 = JSON 1줄
    # force_ascii=False → 한글 그대로 저장
    # -----------------------------------------------------
    df.to_json(
        jsonl_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print()
    print("=" * 60)
    print("수집 완료")
    print("=" * 60)

    print(
        f"총 게시글: {len(df)}개"
    )

    print(
        f"CSV   : {csv_path}"
    )

    print(
        f"PKL   : {pkl_path}"
    )

    print(
        f"JSONL : {jsonl_path}"
    )

    return df

# =========================================================
# 9. 실행 환경 정보 출력
# =========================================================

def print_config():

    print()
    print("=" * 60)
    print("네이버 카페 크롤링 설정")
    print("=" * 60)

    print(
        f"담당자: {COLLECTOR_NAME}"
    )

    print(
        f"카페: {CAFE_NAME}"
    )

    print(
        f"CAFE_ID: {CAFE_ID}"
    )

    print(
        f"MENU_IDS: {MENU_IDS}"
    )

    print(
        f"검색 기간: "
        f"{START_DATE} ~ {END_DATE}"
    )

    print(
        f"검색 키워드 수: "
        f"{len(KEYWORDS)}개"
    )

    print(
        f"MAX_PAGE: {MAX_PAGE}"
    )

    print(
        f"Chrome Profile: "
        f"{CHROME_PROFILE_PATH}"
    )

    print("=" * 60)


# =========================================================
# 10. Main
# =========================================================

def main():

    print_config()

    driver = create_driver()

    try:

        # -------------------------------------------------
        # 1. 네이버 로그인 확인
        # -------------------------------------------------

        check_naver_login(
            driver
        )

        # -------------------------------------------------
        # 2. 게시글 URL 수집
        # -------------------------------------------------

        article_data = collect_article_urls(
            driver
        )

        print()
        print("=" * 60)

        print(
            f"중복 제거 후 "
            f"게시글 URL: "
            f"{len(article_data)}개"
        )

        print("=" * 60)

        # -------------------------------------------------
        # 3. 상세 수집
        # -------------------------------------------------

        rows = collect_all_articles(
            driver,
            article_data
        )

        # -------------------------------------------------
        # 4. 저장
        # -------------------------------------------------

        df = save_results(
            rows
        )

        return df

    finally:

        driver.quit()


# =========================================================
# 11. 실행
# =========================================================

if __name__ == "__main__":

    df = main()


네이버 카페 크롤링 설정
담당자: JKS
카페: 대구맘365
CAFE_ID: 24000254
MENU_IDS: [13]
검색 기간: 20250101 ~ 20260901
검색 키워드 수: 12개
MAX_PAGE: 20
Chrome Profile: C:\selenium_profile\naver_cafe

네이버 로그인 상태 확인

네이버 로그인이 필요합니다.

현재 열린 Selenium Chrome 창에서 네이버에 로그인해주세요.

가능하면 '로그인 상태 유지'를 체크하세요.

로그인 완료 후 이 콘솔로 돌아와 Enter를 눌러주세요.


로그인 완료 후 크롤링을 진행합니다.

[검색 시작] 카페=대구맘365 / 게시판=13 / 검색어=위생
기간=20250101 ~ 20260901
1페이지 완료 / 신규=27 / 누적=27
[종료] 게시판=13 / 검색어=위생 / 2페이지 결과 없음

[검색 시작] 카페=대구맘365 / 게시판=13 / 검색어=청결
기간=20250101 ~ 20260901
1페이지 완료 / 신규=12 / 누적=39
[종료] 게시판=13 / 검색어=청결 / 2페이지 결과 없음

[검색 시작] 카페=대구맘365 / 게시판=13 / 검색어=청소
기간=20250101 ~ 20260901
1페이지 완료 / 신규=50 / 누적=89
2페이지 완료 / 신규=50 / 누적=139
3페이지 완료 / 신규=2 / 누적=141
[종료] 게시판=13 / 검색어=청소 / 4페이지 결과 없음

[검색 시작] 카페=대구맘365 / 게시판=13 / 검색어=세탁
기간=20250101 ~ 20260901
1페이지 완료 / 신규=50 / 누적=191
2페이지 완료 / 신규=9 / 누적=200
[종료] 게시판=13 / 검색어=세탁 / 3페이지 결과 없음

[검색 시작] 카페=대구맘365 / 게시판=13 / 검색어=빨래
기간=20250101 ~ 20260901
1페이지 완료 / 신규=50 / 누적=250
2페이지 완료 / 신규=9 / 누적=259
[종료] 게시판=13 / 검색어=빨래